
# AI Image Upscaler + Image Quality Control Workbench

**Workflow:** เลือกโมเดล → เลือก settings → Upload ภาพ → UPSCALE → QC → Download

ออกแบบสำหรับ Google Colab โดยใช้ Colab Forms / file upload / display native เป็นหลัก ไม่มี Gradio, Streamlit, React หรือ web server.

> **Stock QC note:** Technical QC only — platform acceptance is not guaranteed.
> 
> Notebook นี้ไม่ fake implementation: โมเดลที่ไม่มี checkpoint/official setup ที่ใช้งานได้กับ runtime ปัจจุบันจะแสดงเป็น `Unavailable on current runtime` พร้อมเหตุผล และจะไม่ถูกโหลดเข้า VRAM


In [ ]:

# CELL 1 — Environment + GPU
#@title CELL 1 — Environment + GPU
import os, sys, gc, json, math, time, shutil, subprocess, platform, warnings
from pathlib import Path
from datetime import datetime, timezone

WORK_DIR = Path('/content/AI_Image_Upscaler_QC')
INPUT_DIR = WORK_DIR / 'inputs'
OUTPUT_DIR = WORK_DIR / 'outputs'
CACHE_DIR = WORK_DIR / 'cache'
for d in [INPUT_DIR, OUTPUT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')

def run_cmd(cmd, check=False, quiet=False, cwd=None):
    print(f"$ {cmd}") if not quiet else None
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if (not quiet) and p.stdout:
        print(p.stdout[-4000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{p.stdout}")
    return p.returncode, p.stdout

def get_hardware_info():
    info = {
        'python': sys.version.split()[0],
        'platform': platform.platform(),
        'gpu': 'CPU only',
        'vram_total_gb': 0.0,
        'cuda_available': False,
        'cuda': None,
        'pytorch': None,
    }
    try:
        import torch
        info['pytorch'] = torch.__version__
        info['cuda_available'] = bool(torch.cuda.is_available())
        info['cuda'] = getattr(torch.version, 'cuda', None)
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            info['gpu'] = props.name
            info['vram_total_gb'] = round(props.total_memory / (1024**3), 2)
    except Exception as e:
        info['torch_error'] = str(e)
    return info

HARDWARE = get_hardware_info()
print(json.dumps(HARDWARE, indent=2))

def clear_vram():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

clear_vram()


In [ ]:

# CELL 2 — Dependencies
#@title CELL 2 — Dependencies (core + QC; model-specific deps are lazy-installed only when selected)
import importlib.util, site

def pip_install(packages, extra_args=''):
    cmd = f"{sys.executable} -m pip install {extra_args} {packages}"
    code, out = run_cmd(cmd, quiet=False)
    if code != 0:
        raise RuntimeError(out)

CORE_PACKAGES = [
    'pillow>=10.0.0',
    'numpy',
    'matplotlib',
    'opencv-python-headless',
    'scikit-image',
    'pandas',
    'tqdm',
    'huggingface_hub',
    'safetensors',
    'requests',
]

# IQA-PyTorch provides MUSIQ, NIQE and BRISQUE in one toolbox.
# If this install fails, QC engine still computes Laplacian/FFT and reports metric errors explicitly.
QC_PACKAGES = ['pyiqa']

pip_install(' '.join(CORE_PACKAGES))
try:
    pip_install(' '.join(QC_PACKAGES))
    PYIQA_AVAILABLE = True
except Exception as e:
    PYIQA_AVAILABLE = False
    print('⚠️ pyiqa install failed. MUSIQ/NIQE/BRISQUE will be reported as unavailable, but Laplacian/FFT QC will still run.')
    print(str(e)[-2000:])

# Imports after install
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, ImageFont
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

print('Core dependencies ready.')


In [ ]:

# CELL 3 — Model Registry
#@title CELL 3 — Extensible Model Registry with verified official sources/checkpoint notes
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional

MODEL_REGISTRY = {
    # Implemented / lazy-loaded
    'Real-ESRGAN': {
        'family': 'GENERAL / GAN',
        'status': 'available',
        'implementation': 'python_api',
        'official_repo': 'https://github.com/xinntao/Real-ESRGAN',
        'weights': {
            'x4': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
            'x2': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
        },
        'native_scales': [2, 4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN'],
        'supports_generative_detail': False,
        'notes': 'Official xinntao Real-ESRGAN weights. 8x uses 4x native + safe resize, not native 8x.',
    },
    'Real-ESRGAN Anime': {
        'family': 'ANIME / ILLUSTRATION',
        'status': 'available',
        'implementation': 'python_api',
        'official_repo': 'https://github.com/xinntao/Real-ESRGAN',
        'weights': {
            'x4': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
        },
        'native_scales': [4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN'],
        'supports_generative_detail': False,
        'notes': 'Official anime 6B x4 checkpoint.',
    },
    '4x-UltraSharp': {
        'family': 'GENERAL / GAN',
        'status': 'available_if_downloadable',
        'implementation': 'esrgan_rrdb_hf',
        'official_repo': 'https://huggingface.co/Kim2091/UltraSharp',
        'weights': {'x4_hf_repo': 'Kim2091/UltraSharp', 'filename': '4x-UltraSharp.pth'},
        'native_scales': [4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN'],
        'supports_generative_detail': False,
        'notes': 'Community ESRGAN/RRDB checkpoint hosted on Hugging Face; notebook verifies availability before loading.',
    },
    'AuraSR': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'available',
        'implementation': 'aura_sr',
        'official_repo': 'https://github.com/fal-ai/aura-sr',
        'weights': {'hf_model': 'fal-ai/AuraSR'},
        'native_scales': [4],
        'supports_tile': False,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None'],
        'supports_generative_detail': True,
        'notes': 'GAN/GigaGAN-derived SR package for AI-generated images. 4x only; overlapped mode reduces seams.',
    },

    # Registered but guarded: no fake implementation.
    'Real-CUGAN': {
        'family': 'ANIME / ILLUSTRATION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/bilibili/ailab/tree/main/Real-CUGAN',
        'native_scales': [2,3,4],
        'reason': 'Official project provides models/tools, but Colab CUDA/PyTorch notebook integration is not stable without separate runtime-specific setup. Not loaded.',
    },
    'SwinIR': {
        'family': 'TRANSFORMER',
        'status': 'unavailable',
        'official_repo': 'https://github.com/JingyunLiang/SwinIR',
        'native_scales': [4],
        'reason': 'Official code/checkpoints exist, but dependency/API drift with current Colab is isolated instead of being mixed into this tool by default.',
    },
    'HAT': {
        'family': 'TRANSFORMER',
        'status': 'unavailable',
        'official_repo': 'https://github.com/XPixelGroup/HAT',
        'native_scales': [4],
        'reason': 'Official checkpoints are primarily Google Drive/Baidu and BasicSR-version sensitive. Not auto-loaded to avoid breaking available models.',
    },
    'Real-HAT': {
        'family': 'TRANSFORMER',
        'status': 'unavailable',
        'official_repo': 'https://github.com/XPixelGroup/HAT',
        'native_scales': [4],
        'reason': 'Runtime-specific BasicSR/checkpoint setup required. Not loaded.',
    },
    'DAT': {
        'family': 'TRANSFORMER',
        'status': 'unavailable',
        'official_repo': 'https://github.com/zhengchen1999/DAT',
        'native_scales': [2,3,4],
        'reason': 'Official checkpoints exist, but package integration is BasicSR-version sensitive; kept isolated.',
    },
    'SUPIR': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/Fanghua-Yu/SUPIR',
        'native_scales': [1,2,4],
        'reason': 'Requires SDXL/SUPIR checkpoints and high VRAM. Current turnkey Colab path would need user-managed model files, so disabled.',
    },
    'OSEDiff': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/cswry/OSEDiff',
        'native_scales': [4],
        'reason': 'Diffusion dependencies/checkpoints are not safely packaged for this single-notebook runtime yet.',
    },
    'TSD-SR': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/Microtreei/TSD-SR',
        'native_scales': [4],
        'reason': 'Requires SD3 model path plus LoRA/prompt embeddings. Disabled because it would require manual checkpoint paths.',
    },
    'InvSR': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/zsyOAOA/InvSR',
        'native_scales': [4],
        'reason': 'Requires SD-Turbo plus InvSR checkpoint; not included to avoid hidden downloads and VRAM failures.',
    },
    'Anime4K': {
        'family': 'ANIME / ILLUSTRATION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/bloc97/Anime4K',
        'native_scales': [2,4,8],
        'reason': 'Shader/libplacebo/Vulkan workflow is not a native Colab image notebook path. Not loaded.',
    },
    'CodeFormer': {
        'family': 'FACE RESTORATION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/sczhou/CodeFormer',
        'native_scales': [1,2],
        'reason': 'Face-restoration-only model; registered as optional face module but not auto-installed to avoid dependency conflicts. Use GFPGAN in this notebook.',
    },
    'GFPGAN': {
        'family': 'FACE RESTORATION',
        'status': 'available_as_face_restoration',
        'official_repo': 'https://github.com/TencentARC/GFPGAN',
        'native_scales': [1],
        'reason': 'Available as post-upscale face restoration option, not a full-image SR model.',
    },
}

AVAILABLE_MAIN_MODELS = [k for k,v in MODEL_REGISTRY.items() if v.get('implementation') in ['python_api','esrgan_rrdb_hf','aura_sr']]

def show_table(rows, title=None, max_colwidth=72):
    """Plain-text tables avoid Colab's verbose dataframe HTML/CSS output."""
    if title:
        print(f'\n{title}')
        print('=' * len(title))
    df = pd.DataFrame(rows)
    with pd.option_context('display.max_colwidth', max_colwidth, 'display.width', 220, 'display.max_columns', 20):
        print(df.to_string(index=False))

registry_rows = []
for name, meta in MODEL_REGISTRY.items():
    registry_rows.append({
        'Model': name,
        'Family': meta.get('family'),
        'Status': meta.get('status'),
        'Native scales': ','.join(map(str, meta.get('native_scales', []))),
        'Reason/Notes': meta.get('reason', meta.get('notes','')),
    })
show_table(registry_rows, title='Model Registry')
print('\nAvailable main models:', ', '.join(AVAILABLE_MAIN_MODELS))


In [ ]:

# CELL 4 — Main User Interface / Controls
#@title CELL 4 — Main User Interface / Controls
MODEL = "Real-ESRGAN" #@param ["Real-ESRGAN", "Real-ESRGAN Anime", "4x-UltraSharp", "AuraSR", "Real-CUGAN", "SwinIR", "HAT", "Real-HAT", "DAT", "SUPIR", "OSEDiff", "TSD-SR", "InvSR", "Anime4K", "CodeFormer", "GFPGAN"]
SCALE = "4x" #@param ["2x", "4x", "8x"]
QUALITY_PRESET = "Balanced" #@param ["Fast", "Balanced", "High Quality"]
TILE = "Auto" #@param ["Auto", "256", "512", "768", "1024"]
PRECISION = "Auto" #@param ["Auto", "FP16", "BF16", "FP32"]
FACE_RESTORATION = "None" #@param ["None", "CodeFormer", "GFPGAN"]
GENERATIVE_DETAIL = "OFF" #@param ["OFF", "ON"]
PRESERVE_ORIGINAL = 0 #@param {type:"slider", min:0, max:100, step:5}
STOCK_QC = False #@param {type:"boolean"}
COMPARE_MODE = False #@param {type:"boolean"}
COMPARE_MODELS = "Real-ESRGAN, 4x-UltraSharp, AuraSR" #@param {type:"string"}

MODEL_SETTINGS = {
    'model': MODEL,
    'scale': int(SCALE.replace('x','')),
    'quality_preset': QUALITY_PRESET,
    'tile': TILE,
    'precision': PRECISION,
    'face_restoration': FACE_RESTORATION,
    'generative_detail': GENERATIVE_DETAIL == 'ON',
    'preserve_original_percent': int(PRESERVE_ORIGINAL),
    'stock_qc': bool(STOCK_QC),
    'compare_mode': bool(COMPARE_MODE),
    'compare_models': [m.strip() for m in COMPARE_MODELS.split(',') if m.strip()],
}

def normalize_settings(settings):
    model = settings['model']
    meta = MODEL_REGISTRY.get(model, {})
    normalized = dict(settings)
    warnings_list = []

    if meta.get('status') == 'unavailable':
        raise RuntimeError(f"{model}: Unavailable on current runtime — {meta.get('reason')}")
    if not meta.get('implementation'):
        raise RuntimeError(f"{model}: Unavailable as a main image upscaler in this notebook — {meta.get('reason', 'No full-image SR implementation is registered.')}")

    if normalized['scale'] not in meta.get('native_scales', []):
        if model in ['Real-ESRGAN', '4x-UltraSharp'] and normalized['scale'] in [2,4,8]:
            warnings_list.append(f"Requested {normalized['scale']}x is not native for {model}; output will use model native scale plus high-quality resize if needed.")
        else:
            native = meta.get('native_scales')
            raise RuntimeError(f"{model} supports native scale(s) {native}, not {normalized['scale']}x.")

    if not meta.get('supports_tile', False):
        normalized['tile'] = 'Not supported'
    if normalized['precision'] == 'BF16' and 'BF16' not in meta.get('supports_precision', []):
        warnings_list.append(f"{model} does not support BF16 in this notebook; using Auto.")
        normalized['precision'] = 'Auto'
    if normalized['face_restoration'] not in meta.get('supports_face_restoration', ['None']):
        warnings_list.append(f"Face restoration {normalized['face_restoration']} is not supported with {model}; using None.")
        normalized['face_restoration'] = 'None'
    if normalized['generative_detail'] and not meta.get('supports_generative_detail', False):
        warnings_list.append(f"Generative Detail is not supported by {model}; ignored.")
        normalized['generative_detail'] = False
    return normalized, warnings_list

try:
    PREVIEW_MODEL = MODEL_SETTINGS['compare_models'][0] if MODEL_SETTINGS.get('compare_mode') and MODEL_SETTINGS.get('compare_models') else MODEL
    PREVIEW_SETTINGS = {**MODEL_SETTINGS, 'model': PREVIEW_MODEL}
    NORMALIZED_SETTINGS, SETTING_WARNINGS = normalize_settings(PREVIEW_SETTINGS)
    title = 'Compare Mode first model capability' if MODEL_SETTINGS.get('compare_mode') else 'Selected model capability'
    display(Markdown(f'### {title}'))
    meta = MODEL_REGISTRY.get(PREVIEW_MODEL, {})
    show_table([{
        'Model': PREVIEW_MODEL,
        'Status': meta.get('status'),
        'Native scales': ','.join(map(str, meta.get('native_scales', []))),
        'Tile': meta.get('supports_tile'),
        'Precision': ','.join(meta.get('supports_precision', [])),
        'Face restoration': ','.join(meta.get('supports_face_restoration', ['None'])),
        'Generative detail': meta.get('supports_generative_detail', False),
    }], title='Model Capability Preview')
    if MODEL_SETTINGS.get('compare_mode'):
        print('Compare Mode models:', ', '.join(MODEL_SETTINGS['compare_models']))
    if SETTING_WARNINGS:
        display(Markdown('### Setting warnings'))
        for w in SETTING_WARNINGS: print('⚠️', w)
except Exception as e:
    NORMALIZED_SETTINGS = None
    SETTING_WARNINGS = [str(e)]
    print('❌', e)

if STOCK_QC:
    print('Stock QC enabled: Technical QC only — platform acceptance is not guaranteed.')


In [ ]:

# CELL 5 — Upload + Upscale Engine
#@title CELL 5 — Upload Image + UPSCALE
RUN_UPSCALE = True #@param {type:"boolean"}

from google.colab import files
from urllib.request import urlretrieve

SUPPORTED_EXTS = {'.png', '.jpg', '.jpeg', '.webp'}
MODEL_CACHE = {}

# ---------- dependency isolation / lazy installers ----------
def ensure_realesrgan_stack():
    if importlib.util.find_spec('realesrgan') and importlib.util.find_spec('gfpgan') and importlib.util.find_spec('basicsr'):
        patch_torchvision_functional_tensor()
        return
    print('Installing Real-ESRGAN/GFPGAN stack for selected model only...')
    print('This uses a Colab-safe BasicSR GitHub commit because the PyPI BasicSR release can fail with newer setuptools/torchvision.')
    # Keep this model-specific so other models/QC are not affected.
    # 1) Build tools first. New Colab images may be strict with legacy setup.py packages.
    pip_install('--upgrade pip setuptools wheel')
    # 2) BasicSR from the official repo commit that fixes torchvision.transforms.functional_tensor import.
    try:
        pip_install('git+https://github.com/XPixelGroup/BasicSR@8d56e3a045f9fb3e1d8872f92ee4a4f07f886b0a', extra_args='--use-pep517')
    except Exception:
        print('BasicSR GitHub install failed, retrying with PyPI + compatibility build flags...')
        pip_install('basicsr', extra_args='--use-pep517')
    # 3) Install Real-ESRGAN related packages. --no-deps avoids pip downgrading/replacing the BasicSR we just fixed.
    pip_install('facexlib')
    pip_install('gfpgan realesrgan', extra_args='--no-deps')
    patch_torchvision_functional_tensor()


def patch_torchvision_functional_tensor():
    # Some BasicSR releases import torchvision.transforms.functional_tensor, removed in newer torchvision.
    try:
        import sys, types
        import torchvision.transforms.functional as F
        mod = types.ModuleType('torchvision.transforms.functional_tensor')
        for name in dir(F):
            setattr(mod, name, getattr(F, name))
        sys.modules['torchvision.transforms.functional_tensor'] = mod
    except Exception as e:
        print('Patch warning:', e)


def ensure_aura_sr():
    if importlib.util.find_spec('aura_sr'):
        return
    print('Installing AuraSR for selected model only...')
    pip_install('aura-sr')

# ---------- image validation ----------
def validate_uploaded_image(path: Path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    w, h = img.size
    mode = img.mode
    has_alpha = mode in ('RGBA','LA') or ('transparency' in img.info)
    size_mb = path.stat().st_size / (1024*1024)
    info = {
        'filename': path.name,
        'path': str(path),
        'resolution': [w, h],
        'aspect_ratio': round(w / h, 6) if h else None,
        'color_mode': mode,
        'alpha_channel': bool(has_alpha),
        'file_size_mb': round(size_mb, 3),
    }
    if path.suffix.lower() not in SUPPORTED_EXTS:
        raise ValueError(f'Unsupported file type: {path.suffix}. Use PNG, JPG/JPEG, or WEBP.')
    if w < 8 or h < 8:
        raise ValueError('Image is too small for upscaling/QC.')
    if size_mb > 80:
        print('⚠️ Large file; Colab memory/VRAM may be stressed. Tiling will be used when supported.')
    return img, info


def upload_one_image():
    print('Upload one PNG/JPG/JPEG/WEBP image...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No image uploaded.')
    if len(uploaded) > 1:
        print(f'⚠️ Multiple files uploaded ({len(uploaded)}). This notebook run will process the first file only. Use Compare Mode for multiple models on one image, or rerun this cell for another image.')
    name = next(iter(uploaded.keys()))
    src = INPUT_DIR / name
    with open(src, 'wb') as f:
        f.write(uploaded[name])
    img, info = validate_uploaded_image(src)
    normalized_path = INPUT_DIR / 'input_normalized.png'
    # Keep alpha when present; otherwise RGB.
    if info['alpha_channel']:
        img.convert('RGBA').save(normalized_path)
    else:
        img.convert('RGB').save(normalized_path)
    info['normalized_path'] = str(normalized_path)
    print('Input image info:')
    print(json.dumps(info, indent=2))
    return normalized_path, info

# ---------- automatic VRAM/precision/tile ----------
def effective_tile(tile_setting, supports_tile=True):
    if not supports_tile:
        return 0
    if tile_setting != 'Auto':
        return int(tile_setting)
    vram = HARDWARE.get('vram_total_gb', 0)
    if vram == 0:
        return 256
    if vram < 8:
        return 256
    if vram < 16:
        return 512
    return 0  # Real-ESRGAN uses 0 for no tiling


def effective_precision(precision):
    import torch
    if precision == 'Auto':
        return 'FP16' if torch.cuda.is_available() else 'FP32'
    if precision == 'FP16' and not torch.cuda.is_available():
        print('⚠️ FP16 requested but CUDA unavailable; using FP32.')
        return 'FP32'
    if precision == 'BF16':
        # Only used by models that support it; otherwise normalized before.
        return 'BF16'
    return precision

# ---------- model runners ----------
def download_url(url, dst: Path, min_bytes=1024*1024, retries=3):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size >= min_bytes:
        return dst
    if dst.exists():
        print(f'Existing file looks incomplete; re-downloading: {dst}')
        dst.unlink(missing_ok=True)
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            print(f'Downloading ({attempt}/{retries}) {url} -> {dst}')
            try:
                import requests
                with requests.get(url, stream=True, timeout=60, headers={'User-Agent': 'Mozilla/5.0'}) as r:
                    r.raise_for_status()
                    with open(dst, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)
            except Exception:
                # Fallback for environments where requests streaming is unavailable.
                urlretrieve(url, dst)
            if dst.exists() and dst.stat().st_size >= min_bytes:
                return dst
            raise RuntimeError(f'Downloaded file is too small or missing: {dst}')
        except Exception as e:
            last_error = e
            dst.unlink(missing_ok=True)
            print(f'⚠️ Download failed: {e}')
            time.sleep(2 * attempt)
    raise RuntimeError(f'Could not download checkpoint after {retries} attempts: {url}\nLast error: {last_error}')


def run_realesrgan_like(input_path: Path, output_path: Path, settings: dict, ultrasharp=False):
    clear_vram()
    ensure_realesrgan_stack()
    patch_torchvision_functional_tensor()
    import torch
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan.archs.srvgg_arch import SRVGGNetCompact
    from huggingface_hub import hf_hub_download

    model_name = settings['model']
    scale_req = int(settings['scale'])
    precision = effective_precision(settings['precision'])
    half = (precision == 'FP16' and torch.cuda.is_available())
    tile = effective_tile(settings['tile'], True)

    if ultrasharp:
        native_scale = 4
        weight_path = hf_hub_download(repo_id='Kim2091/UltraSharp', filename='4x-UltraSharp.pth', cache_dir=str(CACHE_DIR/'hf'))
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    elif model_name == 'Real-ESRGAN Anime':
        native_scale = 4
        weight_path = download_url(MODEL_REGISTRY[model_name]['weights']['x4'], CACHE_DIR/'weights/RealESRGAN_x4plus_anime_6B.pth')
        model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4, act_type='prelu')
    else:
        native_scale = 2 if scale_req == 2 else 4
        key = 'x2' if native_scale == 2 else 'x4'
        weight_path = download_url(MODEL_REGISTRY[model_name]['weights'][key], CACHE_DIR/f'weights/RealESRGAN_{key}.pth')
        if native_scale == 2:
            model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
        else:
            model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)

    upsampler = RealESRGANer(
        scale=native_scale,
        model_path=str(weight_path),
        dni_weight=None,
        model=model,
        tile=tile,
        tile_pad=10,
        pre_pad=0,
        half=half,
        gpu_id=0 if torch.cuda.is_available() else None,
    )

    # RealESRGANer follows the official OpenCV path: input/output arrays are BGR/BGRA.
    # Using PIL RGB arrays here can silently swap red/blue channels in outputs.
    img = cv2.imread(str(input_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise RuntimeError(f'OpenCV could not read input image: {input_path}')
    try:
        output, _ = upsampler.enhance(img, outscale=float(scale_req))
    except RuntimeError as e:
        if 'CUDA out of memory' in str(e) and tile == 0:
            print('⚠️ CUDA OOM; retrying with tile=256...')
            clear_vram()
            upsampler.tile = 256
            output, _ = upsampler.enhance(img, outscale=float(scale_req))
        elif half:
            print('⚠️ FP16 failed; retrying FP32...')
            clear_vram()
            return run_realesrgan_like(input_path, output_path, {**settings, 'precision': 'FP32'}, ultrasharp=ultrasharp)
        else:
            raise
    cv2.imwrite(str(output_path), output)

    del upsampler, model
    clear_vram()
    return {'effective_precision': precision, 'tile': tile, 'native_scale': native_scale}


def run_aurasr(input_path: Path, output_path: Path, settings: dict):
    clear_vram()
    ensure_aura_sr()
    import torch
    from aura_sr import AuraSR
    if int(settings['scale']) != 4:
        raise RuntimeError('AuraSR supports 4x only in its official API.')
    precision = effective_precision(settings['precision'])
    img = Image.open(input_path).convert('RGB')
    try:
        model = AuraSR.from_pretrained('fal-ai/AuraSR')
    except TypeError:
        model = AuraSR.from_pretrained()
    if settings.get('generative_detail'):
        if hasattr(model, 'upscale_4x_overlapped'):
            out = model.upscale_4x_overlapped(img)
        else:
            out = model.upscale_4x(img)
    else:
        out = model.upscale_4x(img)
    out.save(output_path)
    del model
    clear_vram()
    return {'effective_precision': precision, 'tile': 'Not supported', 'native_scale': 4}


def apply_preserve_original(input_path: Path, output_path: Path, preserve_percent: int):
    """Blend the SR result with a high-quality resized original.

    preserve_percent = 0 keeps the model output unchanged.
    preserve_percent = 100 returns a pure Lanczos-resized original.
    This is a conservative fidelity control, especially useful for generative SR.
    """
    try:
        preserve_percent = int(preserve_percent or 0)
    except Exception:
        preserve_percent = 0
    preserve_percent = max(0, min(100, preserve_percent))
    if preserve_percent <= 0:
        return 'OFF'

    sr = Image.open(output_path)
    mode = 'RGBA' if sr.mode == 'RGBA' else 'RGB'
    sr_img = sr.convert(mode)
    original = ImageOps.exif_transpose(Image.open(input_path)).convert(mode)
    original_up = original.resize(sr_img.size, Image.Resampling.LANCZOS)
    alpha = preserve_percent / 100.0
    blended = Image.blend(sr_img, original_up, alpha)
    blended.save(output_path)
    return f'{preserve_percent}% blend with resized original'


def apply_gfpgan_if_requested(image_path: Path, settings: dict):
    if settings.get('face_restoration') != 'GFPGAN':
        return None
    print('Applying GFPGAN face restoration as post-process...')
    ensure_realesrgan_stack()
    patch_torchvision_functional_tensor()
    import torch
    from gfpgan import GFPGANer
    weight = download_url('https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth', CACHE_DIR/'weights/GFPGANv1.3.pth')
    restorer = GFPGANer(model_path=str(weight), upscale=1, arch='clean', channel_multiplier=2, bg_upsampler=None, device='cuda' if torch.cuda.is_available() else 'cpu')
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    _, _, restored = restorer.enhance(img, has_aligned=False, only_center_face=False, paste_back=True)
    cv2.imwrite(str(image_path), restored)
    del restorer
    clear_vram()
    return 'GFPGANv1.3 applied'


def upscale_one_model(input_path: Path, input_info: dict, model_name: str, settings: dict, suffix=''):
    meta = MODEL_REGISTRY.get(model_name, {})
    local_settings, warns = normalize_settings({**settings, 'model': model_name})
    if meta.get('status') == 'unavailable':
        raise RuntimeError(f"{model_name}: Unavailable on current runtime — {meta.get('reason')}")

    out_name = f"upscaled{suffix}.png" if suffix else 'upscaled.png'
    output_path = OUTPUT_DIR / out_name
    print(f"\n===== UPSCALE: {model_name} =====")
    print('Settings:', json.dumps(local_settings, indent=2))
    t0 = time.time()
    if meta.get('implementation') == 'aura_sr':
        proc_info = run_aurasr(input_path, output_path, local_settings)
    elif meta.get('implementation') == 'esrgan_rrdb_hf':
        proc_info = run_realesrgan_like(input_path, output_path, local_settings, ultrasharp=True)
    elif meta.get('implementation') == 'python_api':
        proc_info = run_realesrgan_like(input_path, output_path, local_settings, ultrasharp=False)
    else:
        raise RuntimeError(f'{model_name} has no runnable implementation in this notebook.')

    preserve_note = apply_preserve_original(input_path, output_path, local_settings.get('preserve_original_percent', 0))
    face_note = apply_gfpgan_if_requested(output_path, local_settings)
    elapsed = round(time.time() - t0, 2)
    out_img = Image.open(output_path)
    result = {
        'model': model_name,
        'output_path': str(output_path),
        'settings': local_settings,
        'setting_warnings': warns,
        'processing_info': {**proc_info, 'elapsed_sec': elapsed, 'preserve_original': preserve_note, 'face_restoration': face_note},
        'input_info': input_info,
        'output_resolution': list(out_img.size),
        'errors': [],
    }
    print(f"✅ Done: {output_path} ({out_img.size[0]}x{out_img.size[1]}) in {elapsed}s")
    clear_vram()
    return result

UPSCALE_RESULTS = []
INPUT_IMAGE_PATH = None
INPUT_INFO = None

if RUN_UPSCALE:
    if (not MODEL_SETTINGS.get('compare_mode')) and NORMALIZED_SETTINGS is None:
        raise RuntimeError('Invalid model/settings. Fix CELL 4 selection first.')
    INPUT_IMAGE_PATH, INPUT_INFO = upload_one_image()
    base_settings = dict(MODEL_SETTINGS)  # raw UI settings; each model is normalized independently below.
    selected_models = base_settings['compare_models'] if base_settings.get('compare_mode') else [base_settings['model']]
    if len(selected_models) > 4:
        print(f'⚠️ Compare Mode has {len(selected_models)} models. Running only the first 4 to reduce Colab time/VRAM risk.')
        selected_models = selected_models[:4]
    # Filter and run sequentially; never load all models at once.
    for m in selected_models:
        try:
            suffix = '_' + ''.join(c if c.isalnum() else '_' for c in m) if len(selected_models) > 1 else ''
            res = upscale_one_model(INPUT_IMAGE_PATH, INPUT_INFO, m, base_settings, suffix=suffix)
            UPSCALE_RESULTS.append(res)
        except Exception as e:
            err = {'model': m, 'errors': [str(e)], 'output_path': None, 'settings': {**base_settings, 'model': m}, 'input_info': INPUT_INFO}
            UPSCALE_RESULTS.append(err)
            print(f"❌ {m}: {e}")
        finally:
            clear_vram()
else:
    print('RUN_UPSCALE is False. Set to True and run this cell when ready.')


In [ ]:

# CELL 6 — Quality Control Engine
#@title CELL 6 — Quality Control Engine (MUSIQ, NIQE, BRISQUE, Laplacian, 2D FFT)
import tempfile

IQA_CACHE = {}

def resize_for_iqa(img: Image.Image, max_side=1280):
    img = ImageOps.exif_transpose(img).convert('RGB')
    w, h = img.size
    if max(w, h) > max_side:
        ratio = max_side / max(w, h)
        img = img.resize((int(w*ratio), int(h*ratio)), Image.Resampling.LANCZOS)
    return img


def pyiqa_score(metric_name: str, img: Image.Image):
    tmp_name = None
    try:
        if globals().get('PYIQA_AVAILABLE') is False:
            return None, 'pyiqa was not installed successfully in CELL 2'
        import torch, pyiqa
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        if metric_name not in IQA_CACHE:
            IQA_CACHE[metric_name] = pyiqa.create_metric(metric_name, device=device)
        metric = IQA_CACHE[metric_name]
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
            tmp_name = tmp.name
            resize_for_iqa(img).save(tmp.name)
        val = metric(tmp_name)
        if hasattr(val, 'detach'):
            val = float(val.detach().cpu().numpy().squeeze())
        else:
            val = float(val)
        return round(val, 4), None
    except Exception as e:
        return None, str(e)
    finally:
        if tmp_name:
            try:
                os.unlink(tmp_name)
            except Exception:
                pass


def laplacian_variance(img: Image.Image):
    arr = np.array(img.convert('RGB'))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    return round(float(cv2.Laplacian(gray, cv2.CV_64F).var()), 4)


def fft_analysis(img: Image.Image):
    gray = np.array(resize_for_iqa(img, max_side=1024).convert('L')).astype(np.float32) / 255.0
    h, w = gray.shape
    window = np.outer(np.hanning(h), np.hanning(w)).astype(np.float32)
    f = np.fft.fftshift(np.fft.fft2(gray * window))
    mag = np.log1p(np.abs(f))
    yy, xx = np.indices((h, w))
    cy, cx = h//2, w//2
    r = np.sqrt(((yy-cy)/(h/2))**2 + ((xx-cx)/(w/2))**2)
    total = float(mag.sum() + 1e-8)
    hf_ratio = float(mag[r > 0.55].sum() / total)
    mf_ratio = float(mag[(r > 0.25) & (r <= 0.55)].sum() / total)
    # line energy: ringing/grid/repeated texture often appears as row/column spikes.
    center_band = 3
    row_energy = float(mag[cy-center_band:cy+center_band+1, :].sum() / total)
    col_energy = float(mag[:, cx-center_band:cx+center_band+1].sum() / total)
    line_energy = row_energy + col_energy
    # peak anomaly outside DC region
    outer = mag[r > 0.18]
    peak_z = float((outer.max() - outer.mean()) / (outer.std() + 1e-8)) if outer.size else 0.0
    status = 'PASS'
    reasons = []
    if hf_ratio > 0.18:
        status = 'WARNING'; reasons.append('High-frequency energy is elevated')
    if line_energy > 0.18 or peak_z > 18:
        status = 'WARNING'; reasons.append('Possible periodic/grid or ringing pattern')
    if hf_ratio > 0.25 or line_energy > 0.25 or peak_z > 25:
        status = 'FAIL'; reasons.append('Strong synthetic high-frequency artifact pattern')
    return {
        'status': status,
        'high_frequency_ratio': round(hf_ratio, 5),
        'mid_frequency_ratio': round(mf_ratio, 5),
        'line_energy_ratio': round(line_energy, 5),
        'peak_z': round(peak_z, 3),
        'reasons': reasons or ['No significant FFT artifact detected'],
    }


def compute_quality_metrics(image_path: str):
    img = Image.open(image_path)
    metrics = {}
    errors = {}
    for m in ['musiq', 'niqe', 'brisque']:
        score, err = pyiqa_score(m, img)
        metrics[musiq_name(m)] = score
        if err:
            errors[musiq_name(m)] = err
    metrics['laplacian_variance'] = laplacian_variance(img)
    metrics['fft'] = fft_analysis(img)
    if errors:
        metrics['metric_errors'] = errors
    return metrics


def musiq_name(m):
    return {'musiq':'MUSIQ','niqe':'NIQE','brisque':'BRISQUE'}.get(m, m)


def classify_metric(name, value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return 'WARNING', 'Metric unavailable'
    if name == 'MUSIQ':
        if value >= 65: return 'PASS', 'MUSIQ technical/aesthetic score in pass range'
        if value >= 50: return 'WARNING', 'MUSIQ in warning range'
        return 'FAIL', 'MUSIQ below baseline fail range'
    if name == 'NIQE':
        if 3.0 <= value <= 5.5: return 'PASS', 'NIQE in baseline pass range'
        if 5.5 < value <= 7.0 or value < 3.0: return 'WARNING', 'NIQE outside ideal band; interpret carefully for AI art'
        return 'FAIL', 'NIQE high; likely natural-scene quality degradation'
    if name == 'BRISQUE':
        if value < 30: return 'PASS', 'BRISQUE in pass range'
        if value <= 45: return 'WARNING', 'BRISQUE in warning range'
        return 'FAIL', 'BRISQUE in fail range'
    if name == 'laplacian_variance':
        if 300 <= value <= 1500: return 'PASS', 'Sharpness within target range'
        if 150 <= value < 300 or 1500 < value <= 3000: return 'WARNING', 'Sharpness outside target; possible blur or oversharpening'
        return 'FAIL', 'Sharpness extreme; likely blur or oversharpening/artifacts'
    return 'PASS', ''


def delta_metrics(before, after):
    out = {}
    for k in ['MUSIQ','NIQE','BRISQUE','laplacian_variance']:
        b, a = before.get(k), after.get(k)
        out[k] = None if b is None or a is None else round(a - b, 4)
    try:
        out['fft_high_frequency_ratio'] = round(after['fft']['high_frequency_ratio'] - before['fft']['high_frequency_ratio'], 5)
        out['fft_line_energy_ratio'] = round(after['fft']['line_energy_ratio'] - before['fft']['line_energy_ratio'], 5)
    except Exception:
        pass
    return out


def interpret_quality(before, after, settings):
    reasons, warnings_list, errors = [], [], []
    statuses = []
    delta = delta_metrics(before, after)

    # Raw threshold statuses after upscale
    for k in ['MUSIQ','NIQE','BRISQUE','laplacian_variance']:
        st, reason = classify_metric(k, after.get(k))
        statuses.append(st)
        (errors if st == 'FAIL' else warnings_list if st == 'WARNING' else reasons).append(reason)

    # Direction-aware before/after interpretation — no averaging.
    if delta.get('MUSIQ') is not None:
        if delta['MUSIQ'] > 1: reasons.append('MUSIQ improved')
        elif delta['MUSIQ'] < -3: errors.append('MUSIQ degraded after upscale')
        else: warnings_list.append('MUSIQ changed only slightly')
    if delta.get('NIQE') is not None:
        if delta['NIQE'] < -0.2: reasons.append('NIQE improved')
        elif delta['NIQE'] > 0.5: errors.append('NIQE degraded after upscale')
    if delta.get('BRISQUE') is not None:
        if delta['BRISQUE'] < -1: reasons.append('BRISQUE improved')
        elif delta['BRISQUE'] > 3: errors.append('BRISQUE degraded after upscale')

    lap_b, lap_a = before.get('laplacian_variance'), after.get('laplacian_variance')
    if lap_b and lap_a:
        ratio = lap_a / max(lap_b, 1e-6)
        if lap_a > 3000 or (ratio > 4 and lap_a > 1500):
            errors.append('Sharpness increased excessively; possible oversharpening / edge halos')
        elif ratio > 2.5 and lap_a > 1500:
            warnings_list.append('Sharpness increased strongly; inspect for ringing/halo artifacts')
        elif ratio < 0.65:
            warnings_list.append('Sharpness decreased; possible plastic/smoothed texture')
        else:
            reasons.append('Sharpness change is within expected range')

    fft_after = after.get('fft', {})
    if fft_after.get('status') == 'FAIL':
        errors.extend(fft_after.get('reasons', []))
    elif fft_after.get('status') == 'WARNING':
        warnings_list.extend(fft_after.get('reasons', []))
    else:
        reasons.extend(fft_after.get('reasons', []))

    if delta.get('fft_high_frequency_ratio', 0) > 0.10:
        warnings_list.append('High-frequency energy increased unusually; possible synthetic texture/noise')
    if delta.get('fft_line_energy_ratio', 0) > 0.06:
        warnings_list.append('FFT line energy increased; possible repeated texture/grid/ringing')

    # Generative SR caution
    if MODEL_REGISTRY.get(settings.get('model'), {}).get('supports_generative_detail'):
        warnings_list.append('Generative SR may create new details; added detail is not automatically real detail')

    # Overall status
    if errors or 'FAIL' in statuses:
        status = 'FAIL'
    elif warnings_list or 'WARNING' in statuses:
        status = 'WARNING'
    else:
        status = 'PASS'

    if settings.get('stock_qc'):
        warnings_list.append('Stock QC: Technical QC only — platform acceptance is not guaranteed.')

    return {
        'status': status,
        'reasons': sorted(set(reasons)),
        'warnings': sorted(set(warnings_list)),
        'errors': sorted(set(errors)),
        'delta_metrics': delta,
    }


def create_qc_panel_image(upscaled_path: str, qc_path: str, report: dict):
    img = Image.open(upscaled_path).convert('RGB')
    w, h = img.size
    panel_h = max(260, min(430, int(h * 0.16)))
    canvas = Image.new('RGB', (w, h + panel_h), (245, 245, 245))
    canvas.paste(img, (0, 0))
    draw = ImageDraw.Draw(canvas)
    try:
        font_title = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', max(18, w//60))
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', max(12, w//95))
    except Exception:
        font_title = font = ImageFont.load_default()
    y = h + 16
    status = report['quality_status']
    color = {'PASS': (20, 130, 60), 'WARNING': (210, 140, 0), 'FAIL': (190, 40, 40)}.get(status, (0,0,0))
    draw.rectangle([0, h, w, h + panel_h], fill=(245,245,245))
    draw.text((20, y), f"QC STATUS: {status}", fill=color, font=font_title)
    y += int(panel_h*0.18)
    def fmt_delta(x):
        return 'n/a' if x is None else f'{x:+}'
    lines = [
        f"Model: {report['model']} | Scale: {report['scale']}x | Resolution: {report['input_resolution'][0]}x{report['input_resolution'][1]} -> {report['output_resolution'][0]}x{report['output_resolution'][1]}",
        f"MUSIQ: {report['before_metrics'].get('MUSIQ')} -> {report['after_metrics'].get('MUSIQ')} ({fmt_delta(report['delta_metrics'].get('MUSIQ'))} delta)",
        f"NIQE: {report['before_metrics'].get('NIQE')} -> {report['after_metrics'].get('NIQE')} ({fmt_delta(report['delta_metrics'].get('NIQE'))} delta)",
        f"BRISQUE: {report['before_metrics'].get('BRISQUE')} -> {report['after_metrics'].get('BRISQUE')} ({fmt_delta(report['delta_metrics'].get('BRISQUE'))} delta)",
        f"Laplacian Variance: {report['before_metrics'].get('laplacian_variance')} -> {report['after_metrics'].get('laplacian_variance')} ({fmt_delta(report['delta_metrics'].get('laplacian_variance'))} delta)",
        f"FFT: {report['after_metrics'].get('fft',{}).get('status')} | {', '.join(report['after_metrics'].get('fft',{}).get('reasons',[])[:2])}",
        f"Processing: precision={report.get('precision')} tile={report['processing_information'].get('tile')} gpu={report.get('GPU')} elapsed={report['processing_information'].get('elapsed_sec')}s",
    ]
    for line in lines:
        draw.text((20, y), line[:220], fill=(20,20,20), font=font)
        y += int(panel_h*0.105)
    canvas.save(qc_path)
    return qc_path

QC_REPORTS = []
if not UPSCALE_RESULTS:
    print('No upscale results found. Run CELL 5 first.')
else:
    print('Computing BEFORE metrics once...')
    before_metrics = compute_quality_metrics(str(INPUT_IMAGE_PATH))
    print('Before metrics:', json.dumps(before_metrics, indent=2))
    for res in UPSCALE_RESULTS:
        if res.get('errors') and not res.get('output_path'):
            print(f"Skipping QC for failed model {res.get('model')}: {res.get('errors')}")
            continue
        print(f"\nComputing AFTER metrics for {res['model']}...")
        after_metrics = compute_quality_metrics(res['output_path'])
        interp = interpret_quality(before_metrics, after_metrics, res['settings'])
        report = {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'input_filename': res['input_info']['filename'],
            'input_resolution': res['input_info']['resolution'],
            'output_resolution': res['output_resolution'],
            'model': res['model'],
            'scale': res['settings']['scale'],
            'model_settings': res['settings'],
            'GPU': HARDWARE.get('gpu'),
            'VRAM': HARDWARE.get('vram_total_gb'),
            'CUDA': HARDWARE.get('cuda'),
            'PyTorch': HARDWARE.get('pytorch'),
            'precision': res['processing_info'].get('effective_precision'),
            'before_metrics': before_metrics,
            'after_metrics': after_metrics,
            'delta_metrics': interp['delta_metrics'],
            'quality_status': interp['status'],
            'reasons': interp['reasons'],
            'warnings': list(sorted(set(interp['warnings'] + res.get('setting_warnings', [])))),
            'errors': interp['errors'] + res.get('errors', []),
            'processing_information': res['processing_info'],
            'source_image_info': res['input_info'],
        }
        stem = Path(res['output_path']).stem
        report_path = OUTPUT_DIR / (f'quality_report_{res["model"].replace(" ","_").replace("/","_")}.json' if len(UPSCALE_RESULTS) > 1 else 'quality_report.json')
        qc_path = OUTPUT_DIR / (f'{stem}_QC.png' if len(UPSCALE_RESULTS) > 1 else 'upscaled_QC.png')
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        create_qc_panel_image(res['output_path'], str(qc_path), report)
        report['report_path'] = str(report_path)
        report['qc_image_path'] = str(qc_path)
        report['upscaled_path'] = res['output_path']
        QC_REPORTS.append(report)
        print(f"QC STATUS for {res['model']}: {report['quality_status']}")
        for r in report['reasons']: print('  ✅', r)
        for w in report['warnings']: print('  ⚠️', w)
        for e in report['errors']: print('  ❌', e)
    IQA_CACHE.clear()
    clear_vram()


In [ ]:

# CELL 7 — Result Viewer
#@title CELL 7 — Result Viewer (Original / Upscaled / QC + Before/After Analysis)
if not QC_REPORTS:
    print('No QC reports found.')
else:
    for report in QC_REPORTS:
        display(Markdown(f"## Result: {report['model']} — {report['quality_status']}"))
        orig = Image.open(INPUT_IMAGE_PATH).convert('RGB')
        up = Image.open(report['upscaled_path']).convert('RGB')
        qc = Image.open(report['qc_image_path']).convert('RGB')
        # Display without modifying files
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(orig); axes[0].set_title(f"Original\n{orig.size[0]}x{orig.size[1]}"); axes[0].axis('off')
        axes[1].imshow(up); axes[1].set_title(f"Upscaled\n{up.size[0]}x{up.size[1]}"); axes[1].axis('off')
        axes[2].imshow(qc); axes[2].set_title('QC image with panel'); axes[2].axis('off')
        plt.tight_layout(); plt.show()

        rows = []
        for k in ['MUSIQ','NIQE','BRISQUE','laplacian_variance']:
            b = report['before_metrics'].get(k)
            a = report['after_metrics'].get(k)
            d = report['delta_metrics'].get(k)
            rows.append({'Metric': k, 'Before': b, 'After': a, 'Delta': d})
        rows.append({'Metric': 'FFT status', 'Before': report['before_metrics'].get('fft',{}).get('status'), 'After': report['after_metrics'].get('fft',{}).get('status'), 'Delta': report['delta_metrics'].get('fft_high_frequency_ratio')})
        show_table(rows, title='Before / After Metrics')

        display(Markdown('### Interpretation'))
        print('STATUS:', report['quality_status'])
        if report['reasons']:
            print('\nPASS signals / positive notes:')
            for x in report['reasons']: print('-', x)
        if report['warnings']:
            print('\nWarnings:')
            for x in report['warnings']: print('-', x)
        if report['errors']:
            print('\nErrors / fail signals:')
            for x in report['errors']: print('-', x)

    if len(QC_REPORTS) > 1:
        display(Markdown('## Compare Mode Summary'))
        rows = []
        for r in QC_REPORTS:
            rows.append({
                'Model': r['model'],
                'Resolution': f"{r['output_resolution'][0]}x{r['output_resolution'][1]}",
                'MUSIQ': r['after_metrics'].get('MUSIQ'),
                'NIQE': r['after_metrics'].get('NIQE'),
                'BRISQUE': r['after_metrics'].get('BRISQUE'),
                'Sharpness': r['after_metrics'].get('laplacian_variance'),
                'FFT': r['after_metrics'].get('fft',{}).get('status'),
                'Status': r['quality_status'],
            })
        show_table(rows, title='Compare Mode Summary')
        print('No automatic model recommendation is made. Please choose the model based on your own visual/QC requirements.')


In [ ]:

# CELL 8 — Download
#@title CELL 8 — Download results
CREATE_ZIP = True #@param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = False #@param {type:"boolean"}

import zipfile

if not QC_REPORTS:
    print('No files to download yet.')
else:
    files_to_zip = []
    for r in QC_REPORTS:
        for key in ['upscaled_path','qc_image_path','report_path']:
            p = Path(r[key])
            if p.exists():
                files_to_zip.append(p)
    if CREATE_ZIP:
        zip_path = OUTPUT_DIR / 'upscaled_result.zip'
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
            for p in files_to_zip:
                z.write(p, arcname=p.name)
        print('Created:', zip_path)
        if AUTO_DOWNLOAD_ZIP:
            files.download(str(zip_path))
    print('\nDownload files:')
    for p in files_to_zip:
        print('-', p)
    if CREATE_ZIP:
        print('-', zip_path)
    print('\nTo download manually in Colab: open the Files panel or run files.download(path).')
